# SolarScan — 05 · Explicabilité (Grad-CAM)

Un bon modèle ne suffit pas : il faut **comprendre ce qu'il regarde**. Grad-CAM produit une **carte de chaleur** indiquant les zones de l'image qui ont le plus pesé dans la décision.

**Pourquoi c'est important :** si le modèle prédit *Hot-Spot* en regardant la **zone chaude**, on lui fait confiance. S'il regarde un bord ou le fond, c'est qu'il triche (et il échouera en production). C'est un argument fort à montrer à un recruteur.

> ⚡ Colab GPU, Run all. On ré-entraîne un ResNet-18 compact (~10 min) puis on visualise."

## 0. Setup

In [ ]:
%pip install -q torch torchvision scikit-learn pandas pillow matplotlib

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device :', device)

In [ ]:
DATA_DIR = Path('data') if Path('data').exists() else Path('../data')
SPLITS = Path('splits') if Path('splits').exists() else Path('../splits')

if not (DATA_DIR / 'module_metadata.json').exists():
    import urllib.request, zipfile, shutil
    url = 'https://github.com/RaptorMaps/InfraredSolarModules/raw/master/2020-02-14_InfraredSolarModules.zip'
    urllib.request.urlretrieve(url, 'solar.zip')
    with zipfile.ZipFile('solar.zip') as z:
        z.extractall('tmp')
    shutil.move('tmp/InfraredSolarModules', 'data')
    DATA_DIR = Path('data')

if not (SPLITS / 'train.csv').exists():
    from sklearn.model_selection import train_test_split
    with open(DATA_DIR / 'module_metadata.json', encoding='utf-8') as f:
        meta = json.load(f)
    data = pd.DataFrame([{'filepath': v['image_filepath'], 'classe': v['anomaly_class']}
                         for v in meta.values()])
    tr, tmp = train_test_split(data, test_size=0.30, stratify=data['classe'], random_state=42)
    va, te = train_test_split(tmp, test_size=0.50, stratify=tmp['classe'], random_state=42)
    SPLITS = Path('splits'); SPLITS.mkdir(exist_ok=True)
    tr.to_csv(SPLITS / 'train.csv', index=False)
    va.to_csv(SPLITS / 'val.csv', index=False)
    te.to_csv(SPLITS / 'test.csv', index=False)
print('OK')

## 1. Un modèle compact (ResNet-18)

On ré-entraîne rapidement un ResNet-18 (Grad-CAM est plus simple sur ResNet : la dernière couche conv est `layer4`).

In [ ]:
CLASSES = sorted(pd.read_csv(SPLITS / 'train.csv')['classe'].unique())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Grayscale(3), transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(10),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Grayscale(3), transforms.Resize((224, 224)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class SolarDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path); self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        return self.transform(Image.open(DATA_DIR / row['filepath'])), CLASS_TO_IDX[row['classe']]

train_dl = DataLoader(SolarDataset(SPLITS / 'train.csv', train_tf), batch_size=64, shuffle=True, num_workers=2)
val_dl = DataLoader(SolarDataset(SPLITS / 'val.csv', eval_tf), batch_size=64, num_workers=2)
n_train = len(train_dl.dataset)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
model = model.to(device)

y_train = pd.read_csv(SPLITS / 'train.csv')['classe'].map(CLASS_TO_IDX).values
w = np.sqrt(compute_class_weight('balanced', classes=np.arange(len(CLASSES)), y=y_train))
criterion = nn.CrossEntropyLoss(weight=torch.tensor(w / w.mean(), dtype=torch.float).to(device), label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

for epoch in range(8):
    model.train()
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        criterion(model(x), y).backward()
        optimizer.step()
    model.eval(); preds, gts = [], []
    with torch.no_grad():
        for x, y in val_dl:
            preds += model(x.to(device)).argmax(1).cpu().tolist(); gts += y.tolist()
    print(f'epoch {epoch+1}/8 - val macro-F1 {f1_score(gts, preds, average="macro"):.3f}')

## 2. Grad-CAM — le principe

1. On passe l'image dans le modèle et on note la classe prédite.
2. On **rétro-propage** le score de cette classe jusqu'à la dernière couche convolutive (`layer4`).
3. Les **gradients** disent quels canaux comptent ; on les moyenne pour pondérer les **activations**.
4. La somme pondérée (+ ReLU) donne la **carte de chaleur** : *où* le modèle a regardé.

In [ ]:
def gradcam(model, x, target_layer):
    acts, grads = {}, {}
    h1 = target_layer.register_forward_hook(lambda m, i, o: acts.__setitem__('v', o.detach()))
    h2 = target_layer.register_full_backward_hook(lambda m, gi, go: grads.__setitem__('v', go[0].detach()))
    model.eval()
    out = model(x)
    cls = out.argmax(1).item()
    model.zero_grad()
    out[0, cls].backward()
    h1.remove(); h2.remove()
    a = acts['v'][0]                       # (C, h, w)
    g = grads['v'][0]                      # (C, h, w)
    weights = g.mean(dim=(1, 2))           # (C,)
    cam = torch.relu((weights[:, None, None] * a).sum(0))
    cam = cam / (cam.max() + 1e-8)
    return cls, cam.cpu().numpy()

## 3. Visualisation

Ligne du haut : l'image thermique (vraie classe). Ligne du bas : la même image avec la carte de chaleur Grad-CAM (classe prédite).

In [ ]:
test_df = pd.read_csv(SPLITS / 'test.csv')
# On prend un exemple de quelques classes parlantes
wanted = ['Hot-Spot', 'Diode', 'Offline-Module', 'Vegetation', 'Cracking', 'Shadowing', 'Cell', 'No-Anomaly']
rows = [test_df[test_df['classe'] == c].iloc[0] for c in wanted]

fig, axes = plt.subplots(2, len(rows), figsize=(2 * len(rows), 4.5))
for j, row in enumerate(rows):
    raw = np.array(Image.open(DATA_DIR / row['filepath']).convert('L').resize((224, 224)))
    x = eval_tf(Image.open(DATA_DIR / row['filepath'])).unsqueeze(0).to(device)
    cls, cam = gradcam(model, x, model.layer4[-1])
    axes[0, j].imshow(raw, cmap='inferno', extent=(0, 224, 224, 0))
    axes[0, j].set_title(row['classe'], fontsize=8); axes[0, j].axis('off')
    axes[1, j].imshow(raw, cmap='gray', extent=(0, 224, 224, 0))
    axes[1, j].imshow(cam, cmap='jet', alpha=0.5, extent=(0, 224, 224, 0))
    ok = '✓' if CLASSES[cls] == row['classe'] else '✗'
    axes[1, j].set_title(f'{ok} {CLASSES[cls]}', fontsize=8); axes[1, j].axis('off')
plt.tight_layout()
plt.show()

## 4. Interprétation

Observe les cartes de chaleur :
- Pour **Hot-Spot / Offline-Module**, la chaleur Grad-CAM doit se poser sur la **zone chaude** du module → le modèle regarde au bon endroit. ✅
- Pour **Diode**, elle devrait cibler le **tiers** caractéristique du module.
- Si une carte se concentre sur un **bord** ou le **fond**, c'est un signal d'alerte (le modèle exploite un artefact).

**Conclusion du projet :** on a couvert toute la chaîne — EDA → préparation → baseline → modélisation → amélioration → **explicabilité**. C'est exactement le workflow d'un Data Scientist.